In [14]:
from google import genai

api_key = input("Gemini API key: ")

client = genai.Client(api_key=api_key)
model = "gemini-3.5-flash-lite"

In [15]:
response = client.models.generate_content(
    model=model,
    contents="Say hello in one sentence."
)

print(response.text)

Hello, how can I help you today?


In [16]:
from google.genai import types

system_prompt = "You are a helpful coding assistant."
user_prompt = "What's in this folder?"

response = client.models.generate_content(
    model=model,
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
    ),
)

print(response.text)

It looks like you didn't attach or specify which folder you are talking about! 

Since I am an AI text assistant, I don't have direct access to your local computer unless you are using a specific environment (like a coding workspace or IDE plugin) that shares the file structure with me.

If you are working in a code editor (like VS Code with an extension) or a platform that allows file uploads, please let me know or try uploading/listing the contents of the folder. 

Otherwise, if you can **list the file names** or **paste a directory tree**, I'd be happy to tell you what they are and how to use them!


In [17]:
import os

def see_file_tree(root_dir: str = ".") -> list[str]:
    tree = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames[:] = [
            d for d in dirnames
            if d not in {".git", ".venv", "__pycache__", ".ipynb_checkpoints"}
        ]

        for name in sorted(dirnames + filenames):
            full_path = os.path.join(dirpath, name)
            tree.append(os.path.relpath(full_path, root_dir))

    return tree

In [18]:
see_file_tree(".")

['.python-version',
 'README.md',
 'pyproject.toml',
 'src',
 'uv.lock',
 'workshop.ipynb',
 'src/coding_agent',
 'src/coding_agent/__init__.py']

In [19]:
see_file_tree_description = types.FunctionDeclaration(
    name="see_file_tree",
    description="List files and directories in a directory.",
    parameters={
        "type": "object",
        "properties": {
            "root_dir": {
                "type": "string",
                "description": "The directory to inspect."
            }
        },
        "required": []
    }
)

In [20]:
user_prompt = "What's in this folder?"

response = client.models.generate_content(
    model=model,
    contents=user_prompt,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=[see_file_tree_description]
            )
        ]
    )
)

In [21]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'root_dir': '.'
          },
          id='call_539917',
          name='see_file_tree'
        ),
        thought_signature=b"\x12`\n^\x01i\x14}\x13\xbb6[K\x821\xda\x8c\x85\xb9\xb3'\x9b\xb9\xeb7\xe0s\xbf\x9b_R\xa1\xeb\xa7~\x0f\xf8X\n\x0cV\xb9\xda\x108\xab\xd8R\xb7jT\xcbk\xb8W;=po\x1c\xabJRC\xbf\x1f*%%\xf4\xee\xba\x97\xe3K\xd09#A\xda\x0b\xab\x93\x9d\x11\xf0F\x93\x8b\xf9r2\x17<\xf9\xa5\xe2?"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='AkHGarCxGJKEvdIPq52skAQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=61,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
     

In [22]:
print(response.candidates[0].content.parts)

[Part(
  function_call=FunctionCall(
    args={
      'root_dir': '.'
    },
    id='call_539917',
    name='see_file_tree'
  ),
  thought_signature=b"\x12`\n^\x01i\x14}\x13\xbb6[K\x821\xda\x8c\x85\xb9\xb3'\x9b\xb9\xeb7\xe0s\xbf\x9b_R\xa1\xeb\xa7~\x0f\xf8X\n\x0cV\xb9\xda\x108\xab\xd8R\xb7jT\xcbk\xb8W;=po\x1c\xabJRC\xbf\x1f*%%\xf4\xee\xba\x97\xe3K\xd09#A\xda\x0b\xab\x93\x9d\x11\xf0F\x93\x8b\xf9r2\x17<\xf9\xa5\xe2?"
)]


In [23]:
function_call = next(
    part.function_call
    for part in response.candidates[0].content.parts
    if part.function_call
)

function_call

FunctionCall(
  args={
    'root_dir': '.'
  },
  id='call_539917',
  name='see_file_tree'
)

In [28]:
args = dict(function_call.args)
args

{'root_dir': '.'}

In [29]:
result = see_file_tree(**args)
result

['.python-version',
 'README.md',
 'pyproject.toml',
 'src',
 'uv.lock',
 'workshop.ipynb',
 'src/coding_agent',
 'src/coding_agent/__init__.py']

In [37]:
first_response = response

In [38]:
tool_response = types.Part.from_function_response(
    name=function_call.name,
    response={"result": result},
)

second_response = client.models.generate_content(
    model=model,
    contents=[
        user_prompt,
        first_response.candidates[0].content,
        tool_response,
    ],
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=[see_file_tree_description]
            )
        ]
    ),
)

print(second_response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'root_dir': 'src/coding_agent/src/coding_agent'
          },
          id='call_580954',
          name='see_file_tree'
        ),
        thought_signature=b'\x12`\n^\x01i\x14}\x13\xe6H+ult\xc2iCnt+C\xf8\x1fzZ\xa2\xa0\xda\x1a\x12\xeb\t2\xceh\xac-\x91\x97\xc2\x15\xad\x81\x94\xab\x1b\xde\xa9\xdb\xfe>[\xc9\x81\x9c\xa5\xa9"W\x910\n\x18\x87x\x05\x01fF.f\te\x19{sS\x93\xd7&\xe0\x93\x0c\xd6\xe0L\x982tp\xec\xe4:\t\x9e\x01\xf4'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='IkPGapW1E523vdIP6uvy8Qw' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=30,
  prompt_token_count=155,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<Medi

In [39]:
def read_file(filepath: str) -> str:
    try:
        if filepath == "uv.lock":
            return "you're not allowed to read this file"

        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: file '{filepath}' not found."

In [40]:
print(read_file("pyproject.toml"))

[project]
name = "coding-agent"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
authors = [
    { name = "mykola-lp", email = "mykola.sukhorab@gmail.com" }
]
requires-python = ">=3.12"
dependencies = [
    "google-genai>=2.28.0",
    "jupyter>=1.1.1",
    "openai>=3.26.0",
]

[project.scripts]
coding-agent = "coding_agent:main"

[build-system]
requires = ["uv_build>=0.12.23,<0.13.0"]
build-backend = "uv_build"



In [41]:
read_file_description = types.FunctionDeclaration(
    name="read_file",
    description="Read the contents of a file in the current project.",
    parameters={
        "type": "object",
        "properties": {
            "filepath": {
                "type": "string",
                "description": "Path to the file to read.",
            }
        },
        "required": ["filepath"],
    },
)

In [42]:
tools = types.Tool(
    function_declarations=[
        see_file_tree_description,
        read_file_description,
    ]
)

In [46]:
user_prompt = "What dependencies does this project have?"

In [47]:
chat = client.chats.create(
    model=model,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[tools],
    ),
)

In [48]:
response = chat.send_message(user_prompt)

In [49]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'root_dir': '.'
          },
          id='call_224818',
          name='see_file_tree'
        ),
        thought_signature=b'\x12`\n^\x01i\x14}\x13\xd7\xa1O|\xe7*69\xa2\x1e\xe2\xb5\xc6\x84\xad\x12\xcf\xad\xe4\te\xe00\xc1@\xf5)<\xc1\xfd\xcc\t\xa8\x9f7\xd1\xf8uP\x7f\xbf\x81*\xd90\x1c\x9d\x0e*\x94\xe2\x80\xe3\xb6\x8e\x02\xc6\x04QC\xb9l\x96\x8eW\xa4\x99\xb4\xf1\xa1\x13\x1f|\xad\x10\xfc\xa7\xc5$$kV\xb5\xd5\xf2KeL\xe0'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='2kTGavjYOd2fxN8P17fZoQM' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=129,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaMod

In [50]:
for part in response.parts:
    if part.function_call:
        function_call = part.function_call

        f_name = function_call.name
        args = dict(function_call.args)

        print(f"tool_call: {f_name}({args})")

        if f_name == "see_file_tree":
            result = see_file_tree(**args)
        elif f_name == "read_file":
            result = read_file(**args)
        else:
            result = {"error": f"unknown function {f_name}"}

        print(result)

tool_call: see_file_tree({'root_dir': '.'})
['.python-version', 'README.md', 'pyproject.toml', 'src', 'uv.lock', 'workshop.ipynb', 'src/coding_agent', 'src/coding_agent/__init__.py']


In [51]:
tool_response = types.Part.from_function_response(
    name=f_name,
    response={"result": result},
)

In [52]:
response = chat.send_message(tool_response)

In [54]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'filepath': 'pyproject.toml'
          },
          id='call_563395',
          name='read_file'
        ),
        thought_signature=b"\x12`\n^\x01i\x14}\x13z\x8e>|L\x9du\x01\xf7F\xa0bY \xa0\x04\x04!\xae-\x91i\x1f G\x04\x85\xb5\xa7G\xb0%\r\xd71\xc5\xea\x84\x7f\xf5\xe4\x9a;}\xed\x1a\xd7\xd2\xe40J\x18\x9e\xe3YJh\x8a\xe9/\xf1Xy\xc9\xf5H\x0c\xb9),\x8eq\xbb%\x94\xdd\x8b\x85\xa5\x9b\xa0'6\xeb\xb2\xbc\xd5.w"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='NUXGaoPrKrKKvdIPxtOEmAI' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=19,
  prompt_token_count=217,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: '